# Week 1 — Data Acquisition, Cleaning, and Preprocessing
**Dataset:** Titanic — Machine Learning from Disaster (Kaggle)

This notebook loads the raw Titanic training data, explores it, handles missing values, checks for duplicates/outliers, and produces a model-ready feature set. Full write-up with reasoning and discussion is in `Week_1_Report.docx`.

## 1. Data Acquisition

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/pplonski/datasets-for-start/refs/heads/master/Titanic/train.csv"
df = pd.read_csv(url)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Initial Exploration

In [2]:
print(df.shape)
print(df.info())
print(df.describe(include="all").T)

(891, 12)
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
None
             count unique                      top freq       mean  \
PassengerId  891.0    NaN                      NaN  NaN      446.0   
Survived     891.0    NaN                      NaN  NaN   0.383838   
Pclass       891.0    NaN         

## 3. Missing-Value Analysis

In [3]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"Missing": missing, "Percent": missing_pct})

,Missing,Percent
PassengerId,0,0.00
Survived,0,0.00
Pclass,0,0.00
Name,0,0.00
Sex,0,0.00
Age,177,19.87
SibSp,0,0.00
Parch,0,0.00
Ticket,0,0.00
Fare,0,0.00


## 4. Duplicate and Consistency Checks

In [4]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate PassengerId:", df["PassengerId"].duplicated().sum())

print(df["Sex"].value_counts(dropna=False))
print(df["Embarked"].value_counts(dropna=False))

Duplicate rows: 0
Duplicate PassengerId: 0
Sex
male      577
female    314
Name: count, dtype: int64
Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64


## 5. Missing-Value Treatment
- **Age**: median imputation (robust to skew/extremes)
- **Embarked**: mode imputation (only 2 missing)
- **Cabin**: replaced with a `CabinKnown` indicator instead of guessing values

In [5]:
age_median = df["Age"].median()
df["Age"] = df["Age"].fillna(age_median)

embarked_mode = df["Embarked"].mode()[0]
df["Embarked"] = df["Embarked"].fillna(embarked_mode)

df["CabinKnown"] = df["Cabin"].notna().astype(int)

print("Median age used:", age_median)
print("Remaining missing:\n", df[["Age", "Embarked"]].isna().sum())

Median age used: 28.0
Remaining missing:
 Age         0
Embarked    0
dtype: int64


## 6. Text Normalization

In [6]:
for col in ["Sex", "Embarked"]:
    df[col] = df[col].astype(str).str.strip().str.lower()

## 7. Outlier Detection (IQR Rule)

In [7]:
for col in ["Age", "Fare", "SibSp", "Parch"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    flags = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: Q1={q1:.2f} Q3={q3:.2f} lower={lower:.2f} upper={upper:.2f} flagged={flags}")

Age: Q1=22.00 Q3=35.00 lower=2.50 upper=54.50 flagged=66
Fare: Q1=7.91 Q3=31.00 lower=-26.72 upper=65.63 flagged=116
SibSp: Q1=0.00 Q3=1.00 lower=-1.50 upper=2.50 flagged=46
Parch: Q1=0.00 Q3=0.00 lower=0.00 upper=0.00 flagged=213


## 8. Fare Treatment
Extreme fares are legitimate (not errors), so instead of deleting them we provide a 99th-percentile capped version and a log-transformed alternative.

In [8]:
fare_cap = df["Fare"].quantile(0.99)
df["Fare_capped"] = df["Fare"].clip(upper=fare_cap)
df["Fare_log"] = np.log1p(df["Fare"])

print("99th percentile fare cap:", round(fare_cap, 2))

99th percentile fare cap: 249.01


## 9. Feature Selection and Encoding

In [9]:
model_df = df[[
    "Survived", "Pclass", "Sex", "Age", "SibSp", "Parch",
    "Fare_capped", "CabinKnown", "Embarked"
]].copy()

model_df = pd.get_dummies(
    model_df,
    columns=["Sex", "Embarked"],
    drop_first=True
)

model_df = model_df.astype({
    c: int for c in model_df.select_dtypes(include="bool").columns
})

print("Final shape:", model_df.shape)
print("Final missing values:\n", model_df.isna().sum())

Final shape: (891, 10)
Final missing values:
 Survived       0
Pclass         0
Age            0
SibSp          0
Parch          0
Fare_capped    0
CabinKnown     0
Sex_male       0
Embarked_q     0
Embarked_s     0
dtype: int64


## 10. Descriptive Findings After Cleaning

In [10]:
overall_survival = df["Survived"].mean() * 100
by_sex = df.groupby("Sex")["Survived"].mean() * 100
by_class = df.groupby("Pclass")["Survived"].mean() * 100

print("Overall survival rate: {:.2f}%".format(overall_survival))
print("\nBy sex:\n", by_sex.round(2))
print("\nBy class:\n", by_class.round(2))

Overall survival rate: 38.38%

By sex:
 Sex
female    74.20
male      18.89
Name: Survived, dtype: float64

By class:
 Pclass
1    62.96
2    47.28
3    24.24
Name: Survived, dtype: float64


## 11. Save Cleaned Dataset

In [11]:
model_df.to_csv("titanic_cleaned_week1.csv", index=False)
print("Saved titanic_cleaned_week1.csv")

Saved titanic_cleaned_week1.csv


## Conclusion
The most important lesson from this task is that data cleaning is a series of judgment calls, not just removing anything imperfect. See `Week_1_Report.docx` for the full discussion, including a note on why the model-ready feature set has more duplicate rows (111) than the raw dataset (0) — that's a real, explained finding, not a bug.